<a href="https://colab.research.google.com/github/mihirmaurya31/XAI-for-Phishing-URL-detection/blob/main/phish_url.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Installations of Libraries**



In [ ]:
!pip install tldextract
!pip install python-whois publicsuffix2 python-dateutil


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.4/107.4 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.0/117.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.0/89.0 kB 7.0 MB/s eta 0:00:00


***Base Dataset***

In [ ]:
import pandas as pd

# Load the dataset
df = pd.read_csv("phishing_site_urls.csv")

# Inspect structure
print(df.head())
print(df['Label'].value_counts())

# Map labels: bad -> 1 (phishing), good -> 0 (legit)
df['Label'] = df['Label'].map({'bad': 1, 'good': 0})

# Drop duplicates if any
df = df.drop_duplicates()

# Reset index
df = df.reset_index(drop=True)

print("Dataset shape after cleaning:", df.shape)
print(df.head())



                                                 URL Label
0  nobell.it/70ffb52d079109dca5664cce6f317373782/...   bad
1  www.dghjdgf.com/paypal.co.uk/cycgi-bin/webscrc...   bad
2  serviciosbys.com/paypal.cgi.bin.get-into.herf....   bad
3  mail.printakid.com/www.online.americanexpress....   bad
4  thewhiskeydregs.com/wp-content/themes/widescre...   bad
Label
good    392924
bad     156422
Name: count, dtype: int64
Dataset shape after cleaning: (507210, 2)
                                                 URL  Label
0  nobell.it/70ffb52d079109dca5664cce6f317373782/...    1.0
1  www.dghjdgf.com/paypal.co.uk/cycgi-bin/webscrc...    1.0
2  serviciosbys.com/paypal.cgi.bin.get-into.herf....    1.0
3  mail.printakid.com/www.online.americanexpress....    1.0
4  thewhiskeydregs.com/wp-content/themes/widescre...    1.0


**Adding Benign Urls from Tranco**

In [ ]:
import pandas as pd

# Load Tranco file (no header)
tranco_df = pd.read_csv("tranco_7NZNX.csv", header=None, names=["Rank", "Domain"])

# Create full URLs by prefixing http:// (or https://)
tranco_df["URL"] = "http://" + tranco_df["Domain"]

# Assign label = 0 (benign)
tranco_df["Label"] = 0

# Keep only relevant columns
benign_df = tranco_df[["URL", "Label"]]

print(benign_df.head())
print("Total benign samples:", benign_df.shape[0])


                     URL  Label
0      http://google.com      0
1         http://mail.ru      0
2   http://microsoft.com      0
3    http://facebook.com      0
4  http://googleapis.com      0
Total benign samples: 1000000


**Attributes based on URL**

**Building dataset based on Algorithm 1**
Algorithm 1 Feature extraction process

Input: URLs  Array of URLs.

Input: signs  Array of signs to count.

Output: dataset.csv  Output csv document.

1: i ← 0

2: totalURLs ← lenght(URLs)  Get the number of URLs in array.

3: while i < totalURLs do

4: url ← URLs(i)

5: countsUrl ← getCounts(url, signs)  Get signs and character counts.

6: for substring in splitURL(url) do  Iterate through the sub-strings of URL.

7: countsSubstring ← getCounts(substrings, signs)  Get signs and character counts.

8: end for

9: measuredFeatures ← f etchFeatures(url)  Get features from external services.

10: toCsv(countsUrl, countsSubstring, measuredFeatures)  Append row to csv.

11: i ← i + 1

12: end while

In [ ]:
# === FINAL: Algorithm 1 Full Dataset Feature Extraction ===
import time, concurrent.futures, requests, pandas as pd, re
from urllib.parse import urlparse, urlunparse, quote
from requests.utils import requote_uri

# -----------------------------
# Config
# -----------------------------
CONNECTIONS = 100             # parallel workers
HTTP_CONNECT_TIMEOUT = 2.0    # seconds to connect
HTTP_READ_TIMEOUT = 2.0       # seconds to read
TIMEOUT = (HTTP_CONNECT_TIMEOUT, HTTP_READ_TIMEOUT)
UA = {"User-Agent": "Mozilla/5.0 (compatible; MScFeatureExtractor/1.0)"}
OUTPUT_FILE = "dataset_full.csv"

SIGNS = [".","-","_","/","?","=","@","&","!"," ","~","˜",",","+","*","#","$","%"]
SIGN_LABELS = {
    ".":"dot","-":"hyphen","_":"underline","/":"slash","?":"questionmark","=":"equal",
    "@":"at","&":"and","!":"exclamation"," ":"space","~":"tilde","˜":"tilde",",":"comma",
    "+":"plus","*":"asterisk","#":"hashtag","$":"dollar","%":"percent"
}

# -----------------------------
# Helpers
# -----------------------------
def normalize_url(u: str) -> str:
    s = (str(u) or "").strip()
    if not s: return ""
    if s.startswith("//"): s = "http:" + s
    if "://" not in s: s = "http://" + s
    p = urlparse(s)
    if not p.netloc and p.path:
        p = urlparse(f"{p.scheme}://{p.path}")
    host = p.hostname or ""
    port = p.port
    user = p.username or ""
    pwd  = p.password or ""
    userinfo = ""
    if user:
        userinfo = quote(user, safe="")
        if pwd: userinfo += ":" + quote(pwd, safe="")
        userinfo += "@"
    if ":" in host and not (host.startswith("[") and host.endswith("]")):
        host = f"[{host}]"
    netloc = userinfo + host + (f":{port}" if port else "")
    return requote_uri(urlunparse((p.scheme, netloc, p.path or "", p.params or "", p.query or "", p.fragment or "")))

def splitURL(url: str) -> dict:
    try:
        p = urlparse(normalize_url(url))
        directory = p.path or ""
        return {
            "domain":   p.hostname or "",
            "directory": directory,
            "file":      directory.split("/")[-1] if directory else "",
            "params":    p.query or "",
            "fragment":  p.fragment or "",
        }
    except Exception:
        return {"domain":"","directory":"","file":"","params":"","fragment":""}

def getCounts(text: str, signs: list, scope: str) -> dict:
    s = text if isinstance(text, str) else ""
    out = {}
    tilde_total, done = s.count("~") + s.count("˜"), False
    for ch in signs:
        token = SIGN_LABELS.get(ch, f"u{ord(ch)}"); col = f"qty_{token}_{scope}"
        if ch in ("~","˜"):
            if not done:
                out[col] = tilde_total; done = True
        else:
            out[col] = s.count(ch)
    if scope == "url": out["length_url"] = len(s)
    else: out[f"{scope}_length"] = len(s)
    return out

# -----------------------------
# Parallel HTTP HEAD (Jack’s fast approach)
# -----------------------------
def _head_worker(url: str) -> dict:
    res = {
        "time_response": float("nan"),
        "qty_redirects": 0,
        "http_status": float("nan"),
        "server_header": "",
        "tls_ssl_certificate": 1 if normalize_url(url).lower().startswith("https") else 0,
    }
    try:
        t0 = time.perf_counter()
        r = requests.head(normalize_url(url), headers=UA, timeout=TIMEOUT, allow_redirects=True)
        res["time_response"] = int((time.perf_counter() - t0) * 1000)
        res["qty_redirects"] = len(r.history)
        res["http_status"]   = r.status_code
        res["server_header"] = r.headers.get("Server","")
    except requests.exceptions.Timeout:
        pass
    except Exception:
        pass
    return res

def fetch_features_parallel(urls, max_workers=CONNECTIONS):
    out = [None] * len(urls)
    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as ex:
        futures = {ex.submit(_head_worker, u): idx for idx, u in enumerate(urls)}
        done = 0; total = len(urls); step = max(1, total // 10)
        for fut in concurrent.futures.as_completed(futures):
            idx = futures[fut]
            try:
                out[idx] = fut.result()
            except Exception:
                out[idx] = _head_worker("")
            done += 1
            if done % step == 0 or done == total:
                print(f"  HEAD progress: {done}/{total}")
    return out

# -----------------------------
# Algorithm 1 Runner (full dataset)
# -----------------------------
def run_full_algorithm(input_csv="phishing_site_urls.csv", url_col=None, output_csv=OUTPUT_FILE):
    df = pd.read_csv(input_csv)

    # detect URL column
    if url_col is None:
        for c in ["URL","url","Url","Domain","domain"]:
            if c in df.columns:
                url_col = c; break
        if url_col is None:
            raise ValueError("No URL column found.")

    # Label mapping
    if "Label" in df.columns:
        df["Label"] = df["Label"].map({"bad":1,"good":0}).fillna(df["Label"])

    urls = df[url_col].astype(str).tolist()
    total = len(urls)
    print(f"\n Running Algorithm 1 on full dataset ({total} URLs)...")

    # Step 9 (external): parallel HEADs
    head_feats = fetch_features_parallel(urls, max_workers=CONNECTIONS)

    # Steps 5–8 (URL + substrings)
    rows = []
    i = 0
    while i < total:
        url = urls[i]
        row = {"url": url}
        if "Label" in df.columns:
            row["Label"] = df.loc[i, "Label"]

        # full URL counts
        row.update(getCounts(url, SIGNS, "url"))
        # split and count parts
        for scope, sub in splitURL(url).items():
            row.update(getCounts(sub, SIGNS, scope))
        # cached HEAD results
        row.update(head_feats[i] if head_feats[i] else _head_worker(""))

        rows.append(row)
        if i % max(1, total // 10) == 0 or i == total - 1:
            print(f"  Progress: {i+1}/{total}")
        i += 1

    out_df = pd.DataFrame(rows)
    out_df.to_csv(output_csv, index=False)
    print(f"\n Extracted {total} URLs → {output_csv}")
    display(out_df.head(100))
    return out_df

# -----------------------------
# Run for entire dataset
# -----------------------------
df_full = run_full_algorithm("phishing_site_urls.csv", output_csv="dataset_full.csv")


/tmp/ipython-input-1296916982.py:133: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["Label"] = df["Label"].map({"bad":1,"good":0}).fillna(df["Label"])



🚀 Running Algorithm 1 on full dataset (549360 URLs)...
  HEAD progress: 54936/549360


Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/urllib3/connection.py", line 568, in getresponse
    assert_header_parsing(httplib_response.msg)
  File "/usr/local/lib/python3.12/dist-packages/urllib3/util/response.py", line 88, in assert_header_parsing
    raise HeaderParsingError(defects=defects, unparsed_data=unparsed_data)
urllib3.exceptions.HeaderParsingError: [MissingHeaderBodySeparatorDefect()], unparsed data: 'last modified header response: If-Modified-Since\r\nX-XSS-Protection: mode=block\r\nDate: Wed, 15 Oct 2025 10:26:49 GMT\r\n\r\n'
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/urllib3/connection.py", line 568, in getresponse
    assert_header_parsing(httplib_response.msg)
  File "/usr/local/lib/python3.12/dist-packages/urllib3/util/response.py", line 88, in assert_header_parsing
    raise HeaderParsingError(defects=defects, unparsed_data=unparsed_data)
urllib3.exceptions.HeaderParsingError: [MissingHea

  HEAD progress: 109872/549360
  HEAD progress: 164808/549360
  HEAD progress: 219744/549360
  HEAD progress: 274680/549360
  HEAD progress: 329616/549360


Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/urllib3/connection.py", line 568, in getresponse
    assert_header_parsing(httplib_response.msg)
  File "/usr/local/lib/python3.12/dist-packages/urllib3/util/response.py", line 88, in assert_header_parsing
    raise HeaderParsingError(defects=defects, unparsed_data=unparsed_data)
urllib3.exceptions.HeaderParsingError: [MissingHeaderBodySeparatorDefect()], unparsed data: 'X-Content-Type-Optionx-frame-options : SAMEORIGIN"\r\nx-xss-protection: 0\r\n\r\n'


...

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/urllib3/connection.py", line 568, in getresponse
    assert_header_parsing(httplib_response.msg)
  File "/usr/local/lib/python3.12/dist-packages/urllib3/util/response.py", line 88, in assert_header_parsing
    raise HeaderParsingError(defects=defects, unparsed_data=unparsed_data)
urllib3.exceptions.HeaderParsingError: [MissingHeaderBodySeparatorDefect()], unparsed data: 'powered by: STFU.net\r\nX-Powered-By: ASP.NET\r\nDate: Wed, 15 Oct 2025 11:31:00 GMT\r\n\r\n'


  HEAD progress: 439488/549360


Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/urllib3/connection.py", line 568, in getresponse
    assert_header_parsing(httplib_response.msg)
  File "/usr/local/lib/python3.12/dist-packages/urllib3/util/response.py", line 88, in assert_header_parsing
    raise HeaderParsingError(defects=defects, unparsed_data=unparsed_data)
urllib3.exceptions.HeaderParsingError: [MissingHeaderBodySeparatorDefect()], unparsed data: 'X-Content-Type-Optionx-frame-options : SAMEORIGIN"\r\nx-xss-protection: 0\r\n\r\n'
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/urllib3/connection.py", line 568, in getresponse
    assert_header_parsing(httplib_response.msg)
  File "/usr/local/lib/python3.12/dist-packages/urllib3/util/response.py", line 88, in assert_header_parsing
    raise HeaderParsingError(defects=defects, unparsed_data=unparsed_data)
urllib3.exceptions.HeaderParsingError: [MissingHeaderBodySeparatorDefect()], unparsed data: 'X-C

  HEAD progress: 494424/549360
  HEAD progress: 549360/549360
  Progress: 1/549360
  Progress: 54937/549360
  Progress: 109873/549360
  Progress: 164809/549360
  Progress: 219745/549360
  Progress: 274681/549360
  Progress: 329617/549360
  Progress: 384553/549360
  Progress: 439489/549360
  Progress: 494425/549360
  Progress: 549360/549360

 Extracted 549360 URLs → dataset_full.csv


,url,Label,qty_dot_url,qty_hyphen_url,qty_underline_url,qty_slash_url,qty_questionmark_url,qty_equal_url,qty_at_url,qty_and_url,...,qty_asterisk_fragment,qty_hashtag_fragment,qty_dollar_fragment,qty_percent_fragment,fragment_length,time_response,qty_redirects,http_status,server_header,tls_ssl_certificate
0,nobell.it/70ffb52d079109dca5664cce6f317373782/...,1.0,6,4,4,10,1,4,0,3,...,0,0,0,0,0,NaN,0,NaN,,0
1,www.dghjdgf.com/paypal.co.uk/cycgi-bin/webscrc...,1.0,5,2,1,4,0,2,0,1,...,0,0,0,0,0,NaN,0,NaN,,0
2,serviciosbys.com/paypal.cgi.bin.get-into.herf....,1.0,7,1,0,11,0,0,0,0,...,0,0,0,0,0,NaN,0,NaN,,0
3,mail.printakid.com/www.online.americanexpress....,1.0,6,0,0,2,0,0,0,0,...,0,0,0,0,0,NaN,0,NaN,,0
4,thewhiskeydregs.com/wp-content/themes/widescre...,1.0,1,1,0,10,1,0,0,0,...,0,0,0,0,0,NaN,0,NaN,,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,rsaxena.5gbfree.com/facebook.html,1.0,3,0,0,1,0,0,0,0,...,0,0,0,0,0,NaN,0,NaN,,0
96,steamcommunity-giveaway.my3gb.com,1.0,2,1,0,0,0,0,0,0,...,0,0,0,0,0,158.0,0,200.0,,0
97,steamglfts.h16.ru,1.0,2,0,0,0,0,0,0,0,...,0,0,0,0,0,NaN,0,NaN,,0
98,steamglfts.hut2.ru/,1.0,2,0,0,1,0,0,0,0,...,0,0,0,0,0,1236.0,1,200.0,nginx/1.24.0 (Ubuntu),0


In [ ]:
from google.colab import files
files.download("dataset_full.csv")
#------Dataset for Phishing URL-----#

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ----------------My RAM crashed here so I used C# to extract the features------------------------

# === Algorithm 1 for TRANCO dataset (parallel HTTP + CSV output) ===
'''import time, concurrent.futures, requests, pandas as pd
from urllib.parse import urlparse, urlunparse, quote
from requests.utils import requote_uri

# ------------- config -------------
TRONCO_FILE = "tranco_7NZNX.csv"   # <-- change if needed
OUTPUT_10   = "tranco_dataset_10.csv"
OUTPUT_100  = "tranco_dataset_100.csv"
OUTPUT_FULL = "tranco_dataset_full.csv"

CONNECTIONS = 100             # parallel workers
HTTP_CONNECT_TIMEOUT = 2.0    # seconds to connect
HTTP_READ_TIMEOUT    = 2.0    # seconds to read
TIMEOUT = (HTTP_CONNECT_TIMEOUT, HTTP_READ_TIMEOUT)
UA = {"User-Agent": "Mozilla/5.0 (compatible; MScTrancoExtractor/1.0)"}

SIGNS = [".","-","_","/","?","=","@","&","!"," ","~","˜",",","+","*","#","$","%"]
SIGN_LABELS = {
    ".":"dot","-":"hyphen","_":"underline","/":"slash","?":"questionmark","=":"equal",
    "@":"at","&":"and","!":"exclamation"," ":"space","~":"tilde","˜":"tilde",",":"comma",
    "+":"plus","*":"asterisk","#":"hashtag","$":"dollar","%":"percent"
}

# ------------- helpers -------------
def normalize_url(u: str) -> str:
    s = (str(u) or "").strip()
    if not s: return ""
    if s.startswith("//"): s = "http:" + s
    if "://" not in s: s = "http://" + s
    p = urlparse(s)
    if not p.netloc and p.path:
        p = urlparse(f"{p.scheme}://{p.path}")
    host = p.hostname or ""
    port = p.port
    user = p.username or ""
    pwd  = p.password or ""
    userinfo = ""
    if user:
        userinfo = quote(user, safe="")
        if pwd: userinfo += ":" + quote(pwd, safe="")
        userinfo += "@"
    if ":" in host and not (host.startswith("[") and host.endswith("]")):
        host = f"[{host}]"
    netloc = userinfo + host + (f":{port}" if port else "")
    return requote_uri(urlunparse((p.scheme, netloc, p.path or "", p.params or "", p.query or "", p.fragment or "")))

def splitURL(url: str) -> dict:
    try:
        p = urlparse(normalize_url(url))
        directory = p.path or ""
        return {
            "domain":   p.hostname or "",
            "directory": directory,
            "file":      directory.split("/")[-1] if directory else "",
            "params":    p.query or "",
            "fragment":  p.fragment or "",
        }
    except Exception:
        return {"domain":"","directory":"","file":"","params":"","fragment":""}

def getCounts(text: str, signs: list, scope: str) -> dict:
    s = text if isinstance(text, str) else ""
    out = {}
    tilde_total, done = s.count("~") + s.count("˜"), False
    for ch in signs:
        token = SIGN_LABELS.get(ch, f"u{ord(ch)}"); col = f"qty_{token}_{scope}"
        if ch in ("~","˜"):
            if not done:
                out[col] = tilde_total; done = True
        else:
            out[col] = s.count(ch)
    if scope == "url": out["length_url"] = len(s)
    else: out[f"{scope}_length"] = len(s)
    return out

# ------------- parallel HTTP HEAD -------------
def _head_worker(url: str) -> dict:
    res = {
        "time_response": float("nan"),
        "qty_redirects": 0,
        "http_status": float("nan"),
        "server_header": "",
        "tls_ssl_certificate": 1 if normalize_url(url).lower().startswith("https") else 0,
    }
    try:
        t0 = time.perf_counter()
        r = requests.head(normalize_url(url), headers=UA, timeout=TIMEOUT, allow_redirects=True)
        res["time_response"] = int((time.perf_counter() - t0) * 1000)
        res["qty_redirects"] = len(r.history)
        res["http_status"]   = r.status_code
        res["server_header"] = r.headers.get("Server","")
    except requests.exceptions.Timeout:
        pass
    except Exception:
        pass
    return res

def fetch_features_parallel(urls, max_workers=CONNECTIONS):
    out = [None] * len(urls)
    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as ex:
        futures = {ex.submit(_head_worker, u): idx for idx, u in enumerate(urls)}
        done = 0; total = len(urls); step = max(1, total // 10)
        for fut in concurrent.futures.as_completed(futures):
            idx = futures[fut]
            try:
                out[idx] = fut.result()
            except Exception:
                out[idx] = _head_worker("")
            done += 1
            if done % step == 0 or done == total:
                print(f"  HEAD progress: {done}/{total}")
    return out

# ------------- Algorithm 1 runner for Tranco -------------
def run_tranco_algorithm(tranco_csv=TRONCO_FILE, limit=None, output_csv="tranco_dataset.csv"):
    # Load tranco (Rank,Domain) -> URL + Label=0
    tdf = pd.read_csv(tranco_csv, header=None, names=["Rank","Domain"])
    tdf["URL"] = "http://" + tdf["Domain"].astype(str)
    tdf["Label"] = 0  # benign
    urls = tdf["URL"].astype(str).tolist()
    if limit is not None: urls = urls[:limit]
    total = len(urls)
    print(f"\n Tranco Algorithm 1 on {total} URLs (parallel HEAD={CONNECTIONS}, timeout={TIMEOUT})")

    # Step 9 (external) in parallel first
    head_feats = fetch_features_parallel(urls, max_workers=CONNECTIONS)

    # Steps 5–8 + TLD extraction
    rows = []
    i = 0
    while i < total:
        url = urls[i]
        row = {"url": url, "Label": 0}

        # URL-wide counts
        row.update(getCounts(url, SIGNS, "url"))

        # substrings (domain, directory, file, params, fragment)
        parts = splitURL(url)
        for scope, sub in parts.items():
            row.update(getCounts(sub, SIGNS, scope))

        # explicit TLD + its length
        dom = parts["domain"]
        row["tld"] = dom.split(".")[-1] if "." in dom else ""
        row["qty_tld_url"] = len(row["tld"])

        # cached HEAD metrics
        row.update(head_feats[i] if head_feats[i] else _head_worker(""))

        rows.append(row)
        if i % max(1, total // 10) == 0 or i == total - 1:
            print(f"  Build progress: {i+1}/{total}")
        i += 1

    out_df = pd.DataFrame(rows)
    out_df.to_csv(output_csv, index=False)
    print(f"\nDone! Wrote {total} rows → {output_csv}")
    display(out_df.head(10))
    return out_df

# ---------------- RUNS ----------------
# First 10
#df_tranco_10  = run_tranco_algorithm(TRONCO_FILE, limit=10,  output_csv=OUTPUT_10)

# First 100
#df_tranco_100 = run_tranco_algorithm(TRONCO_FILE, limit=100, output_csv=OUTPUT_100)

# Full dataset
df_tranco_full = run_tranco_algorithm(TRONCO_FILE, limit=None, output_csv=OUTPUT_FULL)''''''''''



 Tranco Algorithm 1 on 1000000 URLs (parallel HEAD=100, timeout=(2.0, 2.0))


Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/urllib3/connection.py", line 568, in getresponse
    assert_header_parsing(httplib_response.msg)
  File "/usr/local/lib/python3.12/dist-packages/urllib3/util/response.py", line 88, in assert_header_parsing
    raise HeaderParsingError(defects=defects, unparsed_data=unparsed_data)
urllib3.exceptions.HeaderParsingError: [MissingHeaderBodySeparatorDefect()], unparsed data: "X-Content-Type-Options : nosniff\r\nContent-Security-Policy: default-src https:; script-src blob: 'unsafe-inline' 'unsafe-eval' 'self' https://*.uservoice.com https://maps.googleapis.com https://faronics.kayako.com/ https://apis.google.com/; connect-src https: 'self' ws:; img-src blob: https: 'self' data:; style-src 'unsafe-inline' 'self' https://fonts.googleapis.com; font-src https:;\r\nDate: Wed, 15 Oct 2025 16:40:54 GMT\r\n\r\n"
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/urllib3/connection.py", 

  HEAD progress: 100000/1000000


Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/urllib3/connection.py", line 568, in getresponse
    assert_header_parsing(httplib_response.msg)
  File "/usr/local/lib/python3.12/dist-packages/urllib3/util/response.py", line 88, in assert_header_parsing
    raise HeaderParsingError(defects=defects, unparsed_data=unparsed_data)
urllib3.exceptions.HeaderParsingError: [MissingHeaderBodySeparatorDefect()], unparsed data: '404 Not Found: \r\nContent-Encoding: gzip\r\n\r\n'
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/urllib3/connection.py", line 568, in getresponse
    assert_header_parsing(httplib_response.msg)
  File "/usr/local/lib/python3.12/dist-packages/urllib3/util/response.py", line 88, in assert_header_parsing
    raise HeaderParsingError(defects=defects, unparsed_data=unparsed_data)
urllib3.exceptions.HeaderParsingError: [MissingHeaderBodySeparatorDefect()], unparsed data: 'X-Content-Type-Optionx-frame-option

  HEAD progress: 200000/1000000


...

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/urllib3/connection.py", line 568, in getresponse
    assert_header_parsing(httplib_response.msg)
  File "/usr/local/lib/python3.12/dist-packages/urllib3/util/response.py", line 88, in assert_header_parsing
    raise HeaderParsingError(defects=defects, unparsed_data=unparsed_data)
urllib3.exceptions.HeaderParsingError: [MissingHeaderBodySeparatorDefect()], unparsed data: 'Cache-Control header: no-cache\r\nX-Frame-Options: SAMEORIGIN\r\nDate: Wed, 15 Oct 2025 18:48:41 GMT\r\n\r\n'
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/urllib3/connection.py", line 568, in getresponse
    assert_header_parsing(httplib_response.msg)
  File "/usr/local/lib/python3.12/dist-packages/urllib3/util/response.py", line 88, in assert_header_parsing
    raise HeaderParsingError(defects=defects, unparsed_data=unparsed_data)
urllib3.exceptions.HeaderParsingError: [MissingHeaderBodySeparatorDef

  HEAD progress: 800000/1000000


Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/urllib3/connection.py", line 568, in getresponse
    assert_header_parsing(httplib_response.msg)
  File "/usr/local/lib/python3.12/dist-packages/urllib3/util/response.py", line 88, in assert_header_parsing
    raise HeaderParsingError(defects=defects, unparsed_data=unparsed_data)
urllib3.exceptions.HeaderParsingError: [MissingHeaderBodySeparatorDefect()], unparsed data: 'Access-Control-Allow-Origin : *\r\nAccess-Control-Allow-Origin-Methods : X-Requested-With, Content-Type\r\nKeep-Alive: timeout=5, max=100\r\nConnection: Keep-Alive\r\nContent-Type: text/html;charset=utf-8\r\n\r\n'
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/urllib3/connection.py", line 568, in getresponse
    assert_header_parsing(httplib_response.msg)
  File "/usr/local/lib/python3.12/dist-packages/urllib3/util/response.py", line 88, in assert_header_parsing
    raise HeaderParsingError(defects=def

  HEAD progress: 900000/1000000


Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/urllib3/connection.py", line 568, in getresponse
    assert_header_parsing(httplib_response.msg)
  File "/usr/local/lib/python3.12/dist-packages/urllib3/util/response.py", line 88, in assert_header_parsing
    raise HeaderParsingError(defects=defects, unparsed_data=unparsed_data)
urllib3.exceptions.HeaderParsingError: [MissingHeaderBodySeparatorDefect()], unparsed data: 'Referrer-Policy : same-origin\r\nX-Content-Type-Options : nosniff\r\nDate: Wed, 15 Oct 2025 19:28:17 GMT\r\n\r\n'
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/urllib3/connection.py", line 568, in getresponse
    assert_header_parsing(httplib_response.msg)
  File "/usr/local/lib/python3.12/dist-packages/urllib3/util/response.py", line 88, in assert_header_parsing
    raise HeaderParsingError(defects=defects, unparsed_data=unparsed_data)
urllib3.exceptions.HeaderParsingError: [MissingHeaderBodySeparato

**Final merged dataset of the features**

In [ ]:
import pandas as pd

# Load the two datasets
df_phishing = pd.read_csv("dataset_full (2).csv")
df_tranco = pd.read_csv("tranco_dataset_full.csv")

# Concatenate the dataframes
merged_df = pd.concat([df_phishing, df_tranco], ignore_index=True)

# Convert 'Label' column to boolean
merged_df['Label'] = merged_df['Label'].astype(bool)


# Display the first few rows of the merged dataframe
print("Merged dataset shape:", merged_df.shape)
display(merged_df.tail())
merged_df.dtypes.value_counts()
merged_df.dtypes
merged_df["tls_ssl_certificate"].value_counts()

Merged dataset shape: (1549360, 115)


,url,Label,qty_dot_url,qty_hyphen_url,qty_underline_url,qty_slash_url,qty_questionmark_url,qty_equal_url,qty_at_url,qty_and_url,...,qty_asterisk_fragment,qty_hashtag_fragment,qty_dollar_fragment,qty_percent_fragment,fragment_length,time_response,qty_redirects,http_status,server_header,tls_ssl_certificate
1549355,http://jdmtube.com,False,1,0,0,2,0,0,0,0,...,0,0,0,0,0,-1.0,0,-1.0,NaN,0
1549356,http://naaleads.com,False,1,0,0,2,0,0,0,0,...,0,0,0,0,0,1695.0,1,301.0,CloudFront,0
1549357,http://viagra25.us,False,1,0,0,2,0,0,0,0,...,0,0,0,0,0,336.0,1,301.0,cloudflare,0
1549358,http://gortenziya.shop,False,1,0,0,2,0,0,0,0,...,0,0,0,0,0,737.0,1,301.0,nginx-reuseport/1.21.1,0
1549359,http://karatsc.ru,False,1,0,0,2,0,0,0,0,...,0,0,0,0,0,754.0,1,301.0,nginx/1.26.3,0


,count
tls_ssl_certificate,
0,1549351
1,9


In [ ]:
from google.colab import files

# Save the merged dataframe to a CSV file
merged_df.to_csv("final_features.csv", index=False)

# Provide a download link for the CSV file
files.download("final_features.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**Dataset rebuild**

In [ ]:
import pandas as pd

# Load the phishing dataset
df_phishing = pd.read_csv("phishing_site_urls.csv")

# Map labels in phishing dataset: bad -> 1 (phishing), good -> 0 (legit)
df_phishing['Label'] = df_phishing['Label'].map({'bad': 1, 'good': 0})

# Load Tranco dataset (no header)
tranco_df = pd.read_csv("tranco_7NZNX.csv", header=None, names=["Rank", "Domain"])

# Create full URLs by prefixing http://
tranco_df["URL"] = "http://" + tranco_df["Domain"]

# Assign label = 0 (benign) to Tranco dataset
tranco_df["Label"] = 0

# Keep only relevant columns from Tranco dataset
benign_df = tranco_df[["URL", "Label"]]

# Concatenate the dataframes
merged_df = pd.concat([df_phishing, benign_df], ignore_index=True)

# Display the first few rows of the merged dataframe and its shape
print("Merged dataset shape:", merged_df.shape)
display(merged_df.head())

Merged dataset shape: (1549360, 2)


,URL,Label
0,nobell.it/70ffb52d079109dca5664cce6f317373782/...,1.0
1,www.dghjdgf.com/paypal.co.uk/cycgi-bin/webscrc...,1.0
2,serviciosbys.com/paypal.cgi.bin.get-into.herf....,1.0
3,mail.printakid.com/www.online.americanexpress....,1.0
4,thewhiskeydregs.com/wp-content/themes/widescre...,1.0


In [ ]:
from google.colab import files

# Save the merged dataframe to a CSV file
merged_df.to_csv("merged_dataset.csv", index=False)

# Provide a download link for the CSV file
files.download("merged_dataset.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>